# Handwritten Digit Classification Using Gradient Descent

Softmax Regression on MNIST, trained with Gradient Descent implemented from scratch in NumPy.

**Notebook outline:** load data, check the gradients, train, compare learning rates, evaluate, inspect mistakes, save weights for the Streamlit demo.

Math recap (N examples, 10 classes):
- scores `z = Wx + b`, probabilities `p = softmax(z)`
- loss `J = -(1/N) Σ log p[i, y_i]`
- gradients `dJ/dW = (1/N)(P - Y)ᵀX`, `dJ/db = (1/N) Σ(P - Y)`
- update `W ← W - α ∇_W J`, `b ← b - α ∇_b J`

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # so `src` can be imported from notebooks/

import numpy as np
import matplotlib.pyplot as plt

from src.data import load_data
from src.model import SoftmaxClassifier
from src.train import train, lr_sweep, best_by_val_acc
from src.gradcheck import gradient_check
from src.evaluate import (evaluate, plot_loss_curves, plot_accuracy_comparison,
                          plot_confusion_matrix, plot_predictions, plot_weight_templates)

DATASET = os.environ.get("DIGIT_DATASET", "mnist")   # "mnist" is the real project; "digits" is a tiny offline test set
EPOCHS = int(os.environ.get("DIGIT_EPOCHS", 100))
LEARNING_RATES = [0.01, 0.1, 0.5, 2.0, 8.0]
SEED = 0

## 1. Load and prepare the data
Pixels are scaled to 0-1 and flattened to 784 features. MNIST is split into 50,000 train, 10,000 validation and 10,000 test images. The first run downloads MNIST once and caches it in `data/`.

In [ ]:
data = load_data(DATASET, seed=SEED)
print({k: v.shape for k, v in data.items() if k != "image_shape"})

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img, label in zip(axes.ravel(), data["X_train"], data["y_train"]):
    ax.imshow(img.reshape(data["image_shape"]), cmap="gray")
    ax.set_title(f"label {label}", fontsize=8)
    ax.axis("off")
plt.suptitle("Sample training images")
plt.show()

## 2. Verify the gradients
Before training, compare the analytic gradients with numerical ones (central differences). A relative error around 1e-7 or smaller means the derivation and code agree.

In [ ]:
err = gradient_check(data["X_train"], data["y_train"])
print(f"relative error between analytic and numerical gradients: {err:.2e}")
assert err < 1e-6, "Gradients look wrong"

untrained = SoftmaxClassifier(data["X_train"].shape[1], 10, seed=SEED)
print(f"loss of an untrained model: {untrained.loss(data['X_val'], data['y_val']):.4f}   (ln 10 = {np.log(10):.4f})")

## 3. Baseline training run

In [ ]:
model, hist = train(data, lr=0.5, epochs=EPOCHS, seed=SEED, verbose=True)
print(f"validation accuracy: {hist['val_acc'][-1]:.4f}")

## 4. Learning-rate comparison
All runs share the same data split, initial parameters (same seed) and number of epochs, so only the learning rate differs.

In [ ]:
results = lr_sweep(data, LEARNING_RATES, epochs=EPOCHS, seed=SEED)

print(f"{'lr':<8}{'final train loss':<20}{'final val acc':<15}")
for lr, (_, h) in results.items():
    print(f"{lr:<8g}{h['train_loss'][-1]:<20.4f}{h['val_acc'][-1]:<15.4f}")

plot_loss_curves(results, "../results/loss_curves.png")
plot_accuracy_comparison(results, "../results/accuracy_comparison.png")
plt.show()

### Observations (fill in from your own plots)
- **Too small a learning rate:** ...
- **A good learning rate:** ...
- **Too large a learning rate:** (does the loss oscillate, spike or diverge?) ...
- Which learning rate gave the best validation accuracy, and why do you think so?

## 5. Final evaluation on the test set
The learning rate is chosen using validation accuracy only. The test set is used once, at the end.

In [ ]:
best_lr, best_model, _ = best_by_val_acc(results)
test = evaluate(best_model, data["X_test"], data["y_test"])
print(f"best learning rate (by validation accuracy): {best_lr:g}")
print(f"test accuracy: {test['accuracy']:.4f}    test loss: {test['loss']:.4f}")

plot_confusion_matrix(test["confusion_matrix"], "../results/confusion_matrix.png")
plt.show()

In [ ]:
cm = test["confusion_matrix"].copy()
np.fill_diagonal(cm, 0)
top = np.dstack(np.unravel_index(np.argsort(cm.ravel())[::-1][:5], cm.shape))[0]
print("Most common confusions (true -> predicted : count)")
for t, p in top:
    print(f"  {t} -> {p} : {cm[t, p]}")

## 6. Look at predictions and mistakes

In [ ]:
plot_predictions(best_model, data["X_test"], data["y_test"], data["image_shape"], n=16)
plot_predictions(best_model, data["X_test"], data["y_test"], data["image_shape"], n=16, only_wrong=True)
plt.show()

## 7. What did the model learn?
Each row of `W` has one weight per pixel. Red pixels raise the score for that digit, blue pixels lower it.

In [ ]:
plot_weight_templates(best_model, data["image_shape"], "../results/weight_templates.png")
plt.show()

## 8. Save the trained model for the Streamlit demo

In [ ]:
if DATASET == "mnist":
    best_model.save("../results/weights.npz")
    print("saved results/weights.npz  ->  now run:  streamlit run app.py")
else:
    print("Skipped saving: the Streamlit demo needs a model trained on 28x28 MNIST.")

## Conclusions
Write your conclusions here after running the experiments: what you measured, how the learning rate affected convergence, and where the model makes mistakes. Use only the numbers the notebook printed.